# A07 · MCTS with the A2C network: search budget

Score, win rate and self-kill rate of the FC128 network with MCTS, plotted against the number of simulations
per move. There are two match-ups: against the rule-based agent (our standard evaluation) and against the plain
network, i.e. the A2C agent. 0 simulations means the plain network. The last panel shows how long a move
takes, together with the competition's limit of 0.5 s.

A second figure shows T04's short self-play stage: its training curves, and the same evaluation for the
network before and after it.

**Reads** `runs/mcts/` (T04). **Writes** `figures/mcts_search_budget.pdf`, `figures/mcts_search_budget.tex`,
`figures/mcts_selfplay.pdf` and `figures/mcts_selfplay.tex`.

In [ ]:
import json
import pathlib
import shutil

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import seaborn as sns

HERE = pathlib.Path.cwd().resolve()  # run from the notebook's folder or from the repository root
EXP = next(p / "experiments" for p in (HERE, *HERE.parents) if (p / "experiments" / "README.md").exists())
RUN = EXP / "runs" / "mcts"
FIGURES = EXP / "figures"
FIGURES.mkdir(exist_ok=True)

TEXTWIDTH = 16.0 / 2.54
sns.set_theme(context="paper", style="ticks", palette="colorblind", font="serif", rc={
    "text.usetex": shutil.which("latex") is not None,
    "text.latex.preamble": r"\usepackage[T1]{fontenc}\usepackage{amsmath}",
    "font.serif": ["cmr10", "Computer Modern Roman"], "mathtext.fontset": "cm", "axes.formatter.use_mathtext": True,
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9, "legend.fontsize": 8, "legend.title_fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.grid": True, "grid.linewidth": 0.4, "grid.alpha": 0.6,
    "lines.linewidth": 1.1, "figure.dpi": 120, "figure.constrained_layout.use": True,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "pdf.fonttype": 42,
})

## Data

In [ ]:
config = json.loads((RUN / "config.json").read_text())
games = pd.read_csv(RUN / "eval.csv")
if "network" not in games:  # older runs without the self-play stage
    games["network"] = "A2C"
history = pd.read_csv(RUN / "history.csv") if (RUN / "history.csv").exists() else None
NETWORKS = [n for n in ("A2C", "A2C + self-play") if n in set(games.network)]
latency = pd.read_csv(RUN / "latency.csv")
OPPONENTS = ["rule-based", "A2C (bare network)"]
LIMIT = 0.5  # seconds per move

means = games.groupby(["network", "opponent", "simulations"])[["score", "rival_score", "win", "self_kill"]].mean()
milliseconds = 1000 * latency.groupby("simulations").seconds.median()


def row(network, budget):
    rule, a2c = means.loc[(network, "rule-based", budget)], means.loc[(network, "A2C (bare network)", budget)]
    cells = [f"{rule.score:.2f}", f"{rule.rival_score:.2f}", f"{rule.win:.2f}", f"{rule.self_kill:.2f}",
             f"{a2c.score:.2f}", f"{a2c.rival_score:.2f}", f"{a2c.win:.2f}", f"{milliseconds[budget]:.1f}"]
    return " & ".join([str(budget), *cells]) + r" \\"


def table(network, path):
    """The LaTeX table, written by hand because the columns need a two-level header."""
    lines = [r"\begin{tabular}{r rrrr rrr r}", r"\hline",
             r" & \multicolumn{4}{c}{vs.\ rule-based agent} & \multicolumn{3}{c}{vs.\ A2C (bare network)} & \\",
             r"simulations & score & rival & win & self-kill & score & rival & win & latency [ms] \\", r"\hline",
             *[row(network, b) for b in sorted(games.simulations.unique())], r"\hline", r"\end{tabular}"]
    path.write_text("\n".join(lines) + "\n")


table("A2C", FIGURES / "mcts_search_budget.tex")
if "A2C + self-play" in NETWORKS:
    table("A2C + self-play", FIGURES / "mcts_selfplay.tex")
means.unstack(1).round(3)

## Figure

Averages over the evaluation games with 95 % bootstrap intervals. The number of simulations is on a log$_2$ axis,
with 0 at the left edge.

In [ ]:
a2c_games = games[games.network == "A2C"].copy()  # this figure only shows the A2C network
budgets = sorted(games.simulations.unique())
position = {b: i for i, b in enumerate(budgets)}  # evenly spaced: 0, 2, 4, 8, ...
games["x"] = games.simulations.map(position)
a2c_games["x"] = a2c_games.simulations.map(position)
latency["x"] = latency.simulations.map(position)

metrics = {"score": "score", "win": "win rate", "self_kill": "self-kill rate"}
fig, axes = plt.subplots(1, len(metrics) + 1, figsize=(TEXTWIDTH, 0.32 * TEXTWIDTH))
for ax, (column, label) in zip(axes, metrics.items()):
    sns.lineplot(a2c_games, x="x", y=column, hue="opponent", hue_order=OPPONENTS, marker="o", markersize=4,
                 errorbar=("ci", 95), ax=ax, legend=ax is axes[0])
    ax.set(title=label, ylabel=None)
axes[1].set_ylim(0, 1)
axes[2].set_ylim(bottom=0)

sns.lineplot(latency, x="x", y="seconds", estimator="median", errorbar=("pi", 90), marker="o", markersize=4,
             color="0.3", ax=axes[-1])
axes[-1].axhline(LIMIT, color="C3", ls="--", lw=0.8)
axes[-1].set(yscale="log", title="latency per move [s]", ylabel=None)
for ax in axes:
    ax.set_xticks(range(len(budgets)), [str(b) for b in budgets])
    ax.set_xlabel(None)
fig.supxlabel("simulations per move", fontsize=9)
handles, labels = axes[0].get_legend_handles_labels()
axes[0].get_legend().remove()
fig.legend(handles, labels, loc="outside upper center", ncol=2, title="MCTS against", frameon=False)
fig.savefig(FIGURES / "mcts_search_budget.pdf")

## Self-play stage

Top: T04's self-play stage per iteration (losses, the search's value of the current state, and coins and deaths
per game). Bottom: the same evaluation as above for the network before (A2C) and after (A2C + self-play) the stage.

In [ ]:
if history is not None and "A2C + self-play" in NETWORKS:
    fig, axes = plt.subplots(2, 3, figsize=(TEXTWIDTH, 0.55 * TEXTWIDTH))
    top, bottom = axes
    for column, color in (("policy_loss", "C0"), ("value_loss", "C1")):
        top[0].plot(history.iteration, history[column], color=color, label=column.replace("_", " "))
    top[0].set(title="loss", yscale="log")
    top[0].legend(frameon=False, fontsize=7)
    top[1].plot(history.iteration, history.root_value, color="0.3")
    top[1].set(title="root value")
    top[2].plot(history.iteration, history.coins, color="C2", label="coins")
    ax = top[2].twinx()
    ax.plot(history.iteration, history.death, color="C3", label="death rate")
    ax.set_ylim(0, 1)
    ax.grid(False)
    top[2].set(title="coins (green), death rate (red)")
    for a in top:
        a.set_xlabel("self-play iteration")

    for a, (opponent, column, label) in zip(bottom, (("rule-based", "score", "score vs. rule-based"),
                                                     ("rule-based", "win", "win rate vs. rule-based"),
                                                     ("A2C (bare network)", "win", "win rate vs. A2C"))):
        sns.lineplot(games[games.opponent == opponent], x="x", y=column, hue="network", hue_order=NETWORKS,
                     marker="o", markersize=4, errorbar=("ci", 95), ax=a, legend=a is bottom[0])
        a.set(title=label, ylabel=None, xlabel="simulations per move")
        a.set_xticks(range(len(budgets)), [str(b) for b in budgets])
    bottom[1].set_ylim(0, 1)
    bottom[2].set_ylim(0, 1)
    handles, labels = bottom[0].get_legend_handles_labels()
    bottom[0].get_legend().remove()
    fig.legend(handles, labels, loc="outside lower center", ncol=2, frameon=False)
    fig.savefig(FIGURES / "mcts_selfplay.pdf")